# Stage 02a — Preprocessing (review-batch validation & cleanup)

Runs **between** the human review export (`reviewed_patents_<batch>.xlsx`,
produced by the HTML wizard off `01a_wizard_feed`) and `02b_postprocessing`.

Pipeline: **load** the raw wizard export (resolving blank `Image_Path`s
in-memory via `scripts/resolve_image_paths.py` logic — the raw file is never
touched) → **validate & clean** (ghost-block cleanup, `empKin`→`empTilts`
migration, Rule A Combined Thrust, Rule B Fixed Empennage, Rule C
duplicate-chain `UAVSimilar` propagation, Rule D duplicate inheritance,
Rule E `acState` Ground→Other, Rule F booms-X-formation backfill, then a
completeness report) → **human pass** over the flagged queue in the
ipywidgets UI → **export** approved-only images as a brand-new timestamped
`Review_postprocess_<batch>_<timestamp>.xlsx` (formatted per
`Rearranging th eexcell.py`) for `02b_postprocessing`.

Format contract (differs from `excel_schema.py`'s source format!):
7 columns, `Value`s are `"ID — Label"` composites, M3 kinematics are
card-prefixed (`wing1_propKin`, ...), edge tags live in `META/t1EdgeTags`.


## Section 1 — Imports & Config

In [1]:
import sys
from pathlib import Path
from datetime import datetime

repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
import networkx as nx          # Rule C — duplicate-chain connected components
import openpyxl
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
import ipywidgets as widgets
from IPython.display import display, clear_output

from src.config_loader import load_config
import src.processor as proc   # Section 5 — parse_arch_id() for image lookup

cfg = load_config()

sheet_name = "Batch_01"   # <- the batch to preprocess

# ── Run-mode knobs (leave both at their defaults for a normal labelling run) ─
# HEADLESS: True → every interactive widget step is skipped/auto-applied so the
#   whole notebook can execute top-to-bottom unattended ("Run All" / nbconvert
#   smoke test). For real labelling leave it False.
# SMOKE_N_PATENTS: >0 → only the first N patents are loaded (fast pipeline
#   check). 0 = full batch.
import os
HEADLESS = os.environ.get("NB02A_HEADLESS", "0") == "1"
SMOKE_N_PATENTS = int(os.environ.get("NB02A_SMOKE_N", "0"))

print(f"batch: {sheet_name}   |   mode: {'HEADLESS (no widgets)' if HEADLESS else 'interactive'}"
      + (f"   |   SMOKE subset: first {SMOKE_N_PATENTS} patents" if SMOKE_N_PATENTS else ""))


batch: Batch_01   |   mode: interactive


## Section 2 — "Rearranging the excell" layout logic (reusable)

Ports `src/Rearranging th eexcell.py`'s three visual behaviors into one
function we call from Section 6:

1. **Cell merging** — for every column, consecutive rows sharing the same
   value are merged into one cell (vertical-center, wrap-text), exactly as
   the original script's "merge consecutive rows with identical values" step.
2. **Column widths** — the original hardcoded a `{'A': 18, 'B': 10, ...}`
   letter->width dict tied to one fixed 7-column layout (`Patent_ID, Section,
   Sub_Dimension, Field, Value, Source, Image_Path`). Our schema carries more
   columns (`Definition`, `Options`, `Confidence`, `Needs_Review`,
   `pre_process_flags`) and the column order can shift, so a hardcoded letter
   map would silently mis-size or ignore columns. Instead we **auto-fit**:
   each column's width is `max(min_width, min(longest_value + 2, max_width))`,
   computed from that column's actual content.
3. **Text wrap / alignment** — final pass sets `wrap_text=True` on every
   data cell, `vertical='center'` for merged cells and `'top'` otherwise,
   matching the original.

The original script also **truncates** long `Value` cells (`abstract` /
`description_of_drawings`) and `Image_Path` cells for on-screen compactness.
That's a real data loss if baked into the file handed to `02b_postprocessing`,
so it's kept but off by default (`truncate_long_text=False`).


In [2]:
def format_review_workbook(
    df: pd.DataFrame,
    output_xlsx: Path,
    truncate_long_text: bool = False,
    min_col_width: int = 8,
    max_col_width: int = 60,
) -> None:
    """Write df to output_xlsx as TWO sheets:

    - "Review"  — flat, machine-readable, NO cell merging: what
                  02b_postprocessing / proc.load_review_images consume.
                  (Merged cells read back as NaN in pandas — merging the data
                  sheet silently destroys Patent_ID/Value on round-trip.)
    - "Compact" — the human view per src/Rearranging th eexcell.py:
                  consecutive identical cells merged, wrap text, auto-fit.

    Styling uses shared Alignment/Font/Fill objects — constructing one per
    cell makes openpyxl take minutes on a 20k-row batch.
    """
    headers = list(df.columns)
    truncate_fields = {"abstract", "description_of_drawings"}
    header_fill = PatternFill(start_color="F2F2F2", end_color="F2F2F2", fill_type="solid")
    header_font = Font(bold=True)
    align_top = Alignment(vertical="top", wrap_text=True)
    align_center = Alignment(vertical="center", wrap_text=True)

    def _cell_value(row, col_name):
        val = row[col_name]
        if pd.isna(val):
            return None
        val_str = str(val)
        if truncate_long_text:
            if col_name == "Value" and str(row.get("Field")) in truncate_fields and len(val_str) > 25:
                val_str = val_str[:25] + "..."
            if col_name == "Image_Path" and len(val_str) > 15:
                val_str = val_str[:15] + "..."
        return val_str

    def _fill_sheet(ws, merge: bool):
        ws.append(headers)
        for col_num in range(1, len(headers) + 1):
            cell = ws.cell(row=1, column=col_num)
            cell.fill = header_fill
            cell.font = header_font

        for _, row in df.iterrows():
            ws.append([_cell_value(row, c) for c in headers])

        # Wrap-text/top-align every data cell up front (shared object, cheap);
        # merge anchors get re-set to center below.
        for row_cells in ws.iter_rows(min_row=2):
            for cell in row_cells:
                cell.alignment = align_top

        if merge:
            # Merge consecutive rows with identical values, per column.
            for col in range(1, ws.max_column + 1):
                start_row = 2
                for row in range(3, ws.max_row + 1):
                    val_prev = ws.cell(row=row - 1, column=col).value
                    val_curr = ws.cell(row=row, column=col).value
                    if val_curr != val_prev or val_prev is None:
                        if (row - 1) > start_row and val_prev is not None:
                            ws.merge_cells(start_row=start_row, start_column=col, end_row=row - 1, end_column=col)
                            ws.cell(row=start_row, column=col).alignment = align_center
                        start_row = row
                if ws.max_row > start_row and ws.cell(row=start_row, column=col).value is not None:
                    ws.merge_cells(start_row=start_row, start_column=col, end_row=ws.max_row, end_column=col)
                    ws.cell(row=start_row, column=col).alignment = align_center

        # Auto-fit column widths from actual content (the original script's
        # hardcoded {'A': 18, ...} assumed one fixed 7-column layout).
        for col_num, col_name in enumerate(headers, start=1):
            col_letter = get_column_letter(col_num)
            longest = max(
                [len(col_name)] + [len(str(v)) for v in df[col_name].dropna().astype(str)],
                default=len(col_name),
            )
            ws.column_dimensions[col_letter].width = max(min_col_width, min(longest + 2, max_col_width))

    wb = openpyxl.Workbook()
    ws_data = wb.active
    ws_data.title = "Review"
    _fill_sheet(ws_data, merge=False)
    _fill_sheet(wb.create_sheet("Compact"), merge=True)

    wb.save(output_xlsx)
    print(f"Successfully generated formatted workbook (Review + Compact): {output_xlsx}")


## Section 3 — Data Loading

Read the raw `reviewed_patents_<batch>.xlsx` export from the review stage.
Never mutate/overwrite the original export file — all cleaning happens on an
in-memory copy, written out as a new file in Section 6.


In [3]:
EXPORTS_DIR = Path(cfg["paths"]["html_review_exports"])
REVIEWED_XLSX = EXPORTS_DIR / f"reviewed_patents_{sheet_name}.xlsx"

if not REVIEWED_XLSX.exists():
    # Friendly failure: show what IS there, and rescue near-miss filenames
    # (e.g. "reviewed_patents_Batch_05 .xlsx" with a stray space — a real case).
    available = sorted(EXPORTS_DIR.glob("reviewed_patents_*.xlsx"))
    near_miss = [p for p in available
                 if p.name.replace(" ", "") == REVIEWED_XLSX.name.replace(" ", "")]
    if near_miss:
        REVIEWED_XLSX = near_miss[0]
        print(f"⚠ exact filename not found — using near-match {REVIEWED_XLSX.name!r} "
              f"(consider renaming it to remove stray spaces)")
    else:
        batches = [p.stem.removeprefix("reviewed_patents_").strip() for p in available]
        raise FileNotFoundError(
            f"{REVIEWED_XLSX} not found.\n"
            f"  Batches with a wizard export in {EXPORTS_DIR}:\n    "
            + ("\n    ".join(sorted(set(batches))) if batches else "(none)")
            + f"\n  Either set sheet_name (Section 1) to one of those, or export "
            f"{sheet_name} from the wizard ('Export batch' button) first."
        )

# Long format: one row per (Patent_ID, Section, Sub_Dimension, Field, Value,
# Source, Image_Path). NOTE the wizard export is NOT the excel_schema.py
# source format: Values are "ID — Label" composites (use _strip_label), M3
# fields are card-prefixed (wing1_propKin, boom_t1_propKin, ...), and the
# edge-case tags live in META/t1EdgeTags.
xls = pd.ExcelFile(REVIEWED_XLSX)
if "Review" not in xls.sheet_names:
    raise ValueError(
        f"{REVIEWED_XLSX.name} has no 'Review' sheet (found: {xls.sheet_names}). "
        f"The wizard's 'Export batch' writes a flat 'Review' sheet — this file "
        f"looks like something else. Re-export from the wizard."
    )
df_raw = pd.read_excel(xls, sheet_name="Review")

_REQUIRED_COLS = ["Patent_ID", "Section", "Sub_Dimension", "Field", "Value",
                  "Source", "Image_Path"]
_missing_cols = [c for c in _REQUIRED_COLS if c not in df_raw.columns]
if _missing_cols:
    raise ValueError(
        f"{REVIEWED_XLSX.name} 'Review' sheet is missing required column(s) "
        f"{_missing_cols} (has: {list(df_raw.columns)}). This is not a wizard "
        f"batch export — re-export from the wizard."
    )

df = df_raw.copy()   # all downstream work happens on this copy

if SMOKE_N_PATENTS:
    _keep_ids = list(dict.fromkeys(df["Patent_ID"].astype(str)))[:SMOKE_N_PATENTS]
    df = df[df["Patent_ID"].astype(str).isin(_keep_ids)].reset_index(drop=True)
    print(f"SMOKE mode: subset to first {len(_keep_ids)} patent(s), {len(df)} rows")

n_ids = df["Patent_ID"].nunique()
_base_ids = df["Patent_ID"].astype(str).str.split("_arch").str[0]
n_base_patents = _base_ids.nunique()
n_arch_variants = (df["Patent_ID"].astype(str).str.contains("_arch")).sum() and \
    df.loc[df["Patent_ID"].astype(str).str.contains("_arch"), "Patent_ID"].nunique()
print(f"loaded {len(df)} rows / {n_ids} Patent_ID key(s) from {REVIEWED_XLSX.name}")
if n_arch_variants:
    print(f"  ({n_base_patents} distinct patent(s); {n_arch_variants} are "
          f"_archN multi-architecture variant rows on top of their base patent)")

# ── Resolve blank Image_Path cells in-memory (scripts/resolve_image_paths.py) ──
# The wizard's "📎 Attach/Replace Image" can only record a filename, never a
# disk path, so those rows come back with a blank Image_Path. The script
# fixes the file in place; here we apply the same find_image() lookup to the
# in-memory copy instead, so the raw export stays untouched and Section 6's
# output carries resolved paths.
#
# "Image: (fig N)" rows are text-referenced figures with no matched image on
# disk — placeholders by design, skipped. Unresolved files only WARN when
# their block is APPROVED: disapproved blocks (e.g. abandoned clipboard
# pastes like FR3143549A1's "Pasted image*.png") are dropped by Section 6's
# approved-only export anyway, so a missing file there is harmless.
from scripts.resolve_image_paths import find_image

matched_root = Path(cfg["paths"]["matched"])
img_names = df["Sub_Dimension"].astype(str).str.removeprefix("Image: ").str.strip()
is_placeholder = img_names.str.match(r"^\(fig .*\)$") | (img_names == "(none available)")
img_mask = (
    (df["Section"] == "T2")
    & df["Sub_Dimension"].astype(str).str.startswith("Image: ")
    & ~is_placeholder
    & (df["Image_Path"].isna() | (df["Image_Path"].astype(str).str.strip() == ""))
)

# block (Patent_ID, Sub_Dimension) -> approved?
status_rows = df[(df["Section"] == "T2") & (df["Field"] == "status")]
block_approved = {
    (str(r["Patent_ID"]), str(r["Sub_Dimension"])): str(r["Value"]).strip().lower() == "approved"
    for _, r in status_rows.iterrows()
}

resolved, unresolved_approved, unresolved_ignored = 0, [], 0
for idx, row in df[img_mask].iterrows():
    fname = img_names[idx]
    hit = find_image(matched_root, str(row["Patent_ID"]).strip(), fname)
    if hit:
        df.at[idx, "Image_Path"] = str(hit)
        resolved += 1
    elif block_approved.get((str(row["Patent_ID"]), str(row["Sub_Dimension"]))):
        unresolved_approved.append((row["Patent_ID"], fname))
    else:
        unresolved_ignored += 1

print(f"Image_Path resolution: {int(img_mask.sum())} blank (placeholders excluded) | "
      f"{resolved} resolved | {len(unresolved_approved)} unresolved APPROVED | "
      f"{unresolved_ignored} unresolved disapproved (harmless, dropped at export)")
if unresolved_approved:
    print("  ⚠ APPROVED images missing on disk — copy these into the patent's matched/ folder, re-run this cell:")
    for pid, fname in sorted(set(unresolved_approved)):
        print(f"    - {pid}: {fname}")


loaded 25584 rows / 393 Patent_ID key(s) from reviewed_patents_Batch_01.xlsx
  (352 distinct patent(s); 41 are _archN multi-architecture variant rows on top of their base patent)
Image_Path resolution: 45 blank (placeholders excluded) | 39 resolved | 0 unresolved APPROVED | 6 unresolved disapproved (harmless, dropped at export)


## Section 4 — Validation Rule Logic

In [4]:
def _strip_label(value):
    """Wizard export Values are "ID — Label" composites (withLabel format,
    e.g. "TP — Vectored Thrust — Tilt Propulsors"). Return just the ID part;
    None for NaN/blank. Mirrors the HTML's stripLabel().
    """
    if pd.isna(value):
        return None
    return str(value).split(" — ")[0].strip()


def _append_flag(df: pd.DataFrame, patent_ids, message: str) -> None:
    """In-place: append `message` to pre_process_flags for every row whose
    Patent_ID is in patent_ids (idempotent — skips patents that already
    carry that exact message).
    """
    if "pre_process_flags" not in df.columns:
        df["pre_process_flags"] = ""
    df["pre_process_flags"] = df["pre_process_flags"].fillna("")

    mask = df["Patent_ID"].isin(patent_ids)
    df.loc[mask, "pre_process_flags"] = df.loc[mask, "pre_process_flags"].apply(
        lambda existing: existing if message in existing else f"{existing}{message}"
    )


### Rule A — Combined Thrust (flag for review, target = CVT)

"Combined Thrust" **is** the existing G1 option `CVT — Vectored Thrust —
Combined` — no new taxonomy id needed.

For each `Patent_ID`, compare its stripped `topType` (Section `G1`) against
its per-card kinematics rows (`Field` ends with `_propKin`: `wing1_propKin`,
`fuselage_propKin`, `boom_t1_propKin`, ...; stripped ids
`Fixed | Tilt | Vectored | Cyclic`):

- `topType == TP` **and** propKin mixes `Fixed` + tilting → the patent looks
  like combined thrust mislabelled as pure tilt-propulsor. **Flag only**
  (`"Review: Potential Combined Thrust (TP→CVT?);"`) — the Section 5 UI shows
  the image and the reviewer decides via **Update to CVT** / **Keep As Is**.
- `topType == CVT` with mixed propKin is *consistent* — no flag.

The rule deliberately does not auto-overwrite: since the original labels may
simply be wrong, the human pass in Section 5 is the decision point.


In [5]:
CVT_VALUE = "CVT — Vectored Thrust — Combined"   # the wizard's own composite for CVT


def _flag_combined_thrust(df: pd.DataFrame) -> pd.DataFrame:
    """Rule A — Combined Thrust candidates (flag-only; human decides in the
    Section 5 UI whether to switch topType TP → CVT).

    Flags patents whose stripped topType is TP while their *_propKin rows mix
    "Fixed" with a tilting mechanism (Tilt/Vectored/Cyclic). CVT patents with
    mixed propKin are already consistent and are left alone.
    """
    df = df.copy()

    top_type_by_patent = (
        df.loc[df["Field"] == "topType"]
        .set_index("Patent_ID")["Value"].map(_strip_label)
    )

    propkin_rows = df.loc[df["Field"].astype(str).str.endswith("_propKin")]

    def _is_mixed(values: pd.Series) -> bool:
        vals = {_strip_label(v) for v in values} - {None}
        return "Fixed" in vals and bool(vals - {"Fixed"})

    mixed_by_patent = propkin_rows.groupby("Patent_ID")["Value"].apply(_is_mixed)

    flagged_ids = [
        pid for pid, is_mixed in mixed_by_patent.items()
        if is_mixed and top_type_by_patent.get(pid) == "TP"
    ]

    if flagged_ids:
        _append_flag(df, flagged_ids, "Review: Potential Combined Thrust (TP→CVT?);")

    return df


### Step 0b — Empennage Tilt Migration (`empKin` → `empTilts`)

`empKin` (`Fixed | Tilt | Stabilator`) is dropped entirely and replaced by a
single boolean row **only when the empennage actually tilts**:

- stripped `empKin == "Tilt"` → new row `Field="empTilts"`, `Value=True`.
- `Fixed`, `Stabilator`, or no `empKin` row at all → **no row is written** —
  absence means "doesn't tilt," matching how the rest of the schema treats
  booleans (present+True is the only thing that means something).

Must run **before** Rule B, since Rule B's "zero-tilt propulsor → force
empennage to non-tilting" now targets `empTilts` instead of `empKin`.


In [6]:
def _convert_empkin_to_emptilts(df: pd.DataFrame) -> pd.DataFrame:
    """Step 0b — drop empKin, replace with empTilts=True only where it tilts."""
    df = df.copy()

    empkin_rows = df.loc[df["Field"] == "empKin"]
    tilting_ids = [
        pid for pid, val in zip(empkin_rows["Patent_ID"], empkin_rows["Value"])
        if _strip_label(val) == "Tilt"
    ]

    df = df.loc[df["Field"] != "empKin"].reset_index(drop=True)

    if tilting_ids:
        new_rows = pd.DataFrame([
            {
                "Patent_ID": pid, "Section": "M2", "Sub_Dimension": "empKin",
                "Field": "empTilts", "Value": True,
                "Source": "rule_emptilts_migration", "Image_Path": None, "pre_process_flags": "",
            }
            for pid in tilting_ids
        ])
        df = pd.concat([df, new_rows], ignore_index=True)
        _append_flag(df, tilting_ids, "Empennage tilts (empKin migrated to empTilts);")

    print(f"empKin migration: {len(empkin_rows)} empKin row(s) removed, "
          f"{len(tilting_ids)} empTilts=True row(s) added")
    return df


### Rule B — Fixed Empennage

For each `Patent_ID`, if every `propKin` row's Value is `"Fixed"` (i.e. no
tilting/vectored/cyclic mechanism anywhere — pure multirotors/screws,
`topType` `RC`/`MR`, or any winged layout that happens to have none of its
propulsors tilt), the empennage can't legitimately tilt either: any
`empTilts=True` row for that patent is forced to `False` and flagged
`"Empennage tilt forced to False;"`.

Patents with no `empTilts` row at all are already correct (absence already
means "doesn't tilt" — see Step 0b) — nothing to do.


In [7]:
def _flag_fixed_empennage(df: pd.DataFrame) -> pd.DataFrame:
    """Rule B — Fixed Empennage.

    Patents with no tilting propulsor (all *_propKin rows strip to "Fixed")
    get any empTilts=True row forced to False, and are flagged. Patents with
    no empTilts row at all are untouched (already correct).
    """
    df = df.copy()

    propkin_rows = df.loc[df["Field"].astype(str).str.endswith("_propKin")]
    has_tilt_by_patent = propkin_rows.groupby("Patent_ID")["Value"].apply(
        lambda values: any(_strip_label(v) not in (None, "Fixed") for v in values)
    )
    zero_tilt_ids = has_tilt_by_patent[~has_tilt_by_patent].index

    emp_tilts_mask = (
        (df["Field"] == "empTilts")
        & df["Patent_ID"].isin(zero_tilt_ids)
        & (df["Value"] == True)  # noqa: E712 — explicit bool compare, not truthiness
    )
    forced_ids = df.loc[emp_tilts_mask, "Patent_ID"].unique().tolist()

    if forced_ids:
        df.loc[emp_tilts_mask, "Value"] = False
        _append_flag(df, forced_ids, "Empennage tilt forced to False;")

    return df


### Rule C — Duplicate Chain Tagging

The duplicate link lives in the T1 fields `isDuplicate` (bool) +
`duplicateId` (Value = the plain Patent_ID it duplicates — no label suffix).
Chains (A dup-of B dup-of C) are resolved as connected components of an
undirected graph built from those `duplicateId` edges via `networkx`.

The "UAV, but similar enough" tag (`UAVSimilar`) is an **edge-case tag**, not
a disapproval reason: it lives in the `META` section, Field `t1EdgeTags`,
Sub_Dimension `"Review Metadata"`. If any patent in a connected component
carries `UAVSimilar` there, every patent in that component gets it too
(appended comma-separated to an existing t1EdgeTags row, or a new row is
created) and is flagged with `"UAV Tag Propagated;"`.


In [8]:
def _propagate_duplicate_tag(df: pd.DataFrame) -> pd.DataFrame:
    """Rule C — Duplicate Chain Tagging.

    Builds an undirected graph of Patent_ID <-> duplicateId edges, finds
    connected components (chains), and — if any patent in a component carries
    "UAVSimilar" in its META/t1EdgeTags row — stamps that tag onto every
    patent in the component (adding the row if it doesn't already exist).
    """
    df = df.copy()

    graph = nx.Graph()
    graph.add_nodes_from(df["Patent_ID"].astype(str).unique())

    dup_edges = df.loc[
        (df["Field"] == "duplicateId") & df["Value"].notna() & (df["Value"].astype(str).str.strip() != "")
    ]
    for _, row in dup_edges.iterrows():
        graph.add_edge(str(row["Patent_ID"]), str(row["Value"]).strip())

    # "UAV, but similar enough" lives in the META section's t1EdgeTags field
    # (Value contains the tag id "UAVSimilar") — NOT in t1DisapproveReason.
    edge_tag_rows = df["Field"] == "t1EdgeTags"
    tagged_ids = set(
        df.loc[edge_tag_rows & df["Value"].astype(str).str.contains("UAVSimilar", na=False), "Patent_ID"].astype(str)
    )

    # Any chain (component of size > 1) that contains a tagged patent needs
    # the tag propagated to its other members.
    propagate_ids = set()
    for component in nx.connected_components(graph):
        if len(component) > 1 and (component & tagged_ids):
            propagate_ids |= (component - tagged_ids)

    if propagate_ids:
        in_chain = df["Patent_ID"].astype(str).isin(propagate_ids)

        # Existing t1EdgeTags rows: append the tag (comma-separated) unless present.
        existing_mask = edge_tag_rows & in_chain
        df.loc[existing_mask, "Value"] = df.loc[existing_mask, "Value"].apply(
            lambda v: "UAVSimilar" if pd.isna(v) or not str(v).strip()
            else (str(v) if "UAVSimilar" in str(v) else f"{v},UAVSimilar")
        )

        # Patents in the chain with no t1EdgeTags row yet — add one, matching
        # the wizard export's 7-column schema.
        already_updated = set(df.loc[existing_mask, "Patent_ID"].astype(str))
        missing_ids = propagate_ids - already_updated
        if missing_ids:
            new_rows = pd.DataFrame([
                {
                    "Patent_ID": pid, "Section": "META", "Sub_Dimension": "Review Metadata",
                    "Field": "t1EdgeTags", "Value": "UAVSimilar",
                    "Source": "rule_c_propagation", "Image_Path": None, "pre_process_flags": "",
                }
                for pid in missing_ids
            ])
            df = pd.concat([df, new_rows], ignore_index=True)

        _append_flag(df, propagate_ids, "UAV Tag Propagated;")

    return df


### Rule D — Duplicate Inheritance (topType + isApproved from chain root)

The HTML wizard auto-copies M1–M3 to duplicates but **not G1**, and doesn't
force an approval choice on them. Verified on Batch_01: ~131 approved patents
missing `topType` and all 32 NaN-`isApproved` patents are duplicates whose
chain root carries the value.

For every member of a duplicate chain (per Rule C's graph, following
`duplicateId` links to the chain root):

- missing/blank `topType` → copy the root's `topType` (new G1 row,
  `Source = "rule_d_inherited"`), flag `"G1 inherited from duplicate root;"`.
- `isApproved` NaN → copy the root's `isApproved`, flag
  `"Approval inherited from duplicate root;"`.

Members whose root also lacks the value, and non-duplicates missing topType,
are flagged `"Missing topType — needs manual review;"` for the Section 5 UI
instead (nothing to inherit).

The 41 `_archN`-suffixed ids lacking `isApproved` rows are left alone —
approval lives on the base patent by design.


In [9]:
def _resolve_chain_root(patent_id: str, dup_target: pd.Series) -> str:
    """Follow duplicateId links to the chain root (cycle-safe)."""
    seen = set()
    current = str(patent_id)
    while current not in seen:
        seen.add(current)
        target = dup_target.get(current)
        if target is None or pd.isna(target) or not str(target).strip():
            return current
        current = str(target).strip()
    return current  # cycle — return where we stopped


def _inherit_from_duplicate_root(df: pd.DataFrame) -> pd.DataFrame:
    """Rule D — duplicates inherit topType (G1) and isApproved from their
    chain root when they lack a value of their own.

    "Missing topType" is only flagged where it's actually unexpected:
    disapproved patents skip G1 by design, multi-arch patents carry topType
    on their _archN ids (not the base id), and exact duplicates (duplicateType
    "2" — "Images AND aircraft the same") skip G1 by design too: the wizard
    never writes them a topType row, and Section 6's export enforces that they
    stay label-less (the pipeline resolves their full labels via duplicateId).
    """
    df = df.copy()

    dup_rows = df.loc[(df["Field"] == "duplicateId") & df["Value"].notna()
                      & (df["Value"].astype(str).str.strip() != "")]
    dup_target = dup_rows.set_index(dup_rows["Patent_ID"].astype(str))["Value"]
    is_dup_ids = set(dup_target.index)

    top_type_rows = df.loc[df["Field"] == "topType"]
    top_type_by_patent = top_type_rows.set_index(top_type_rows["Patent_ID"].astype(str))["Value"]
    appr_rows = df.loc[df["Field"] == "isApproved"]
    appr_by_patent = appr_rows.set_index(appr_rows["Patent_ID"].astype(str))["Value"]

    # Exact duplicates (type "2") carry no G1 of their own by design — never
    # inherit/flag topType for them, even though they're technically "missing" it.
    dup_type_rows = df.loc[df["Field"] == "duplicateType"]
    dup_type_by_patent = dup_type_rows.set_index(dup_type_rows["Patent_ID"].astype(str))["Value"].map(_strip_label)
    exact_dup_ids = set(dup_type_by_patent[dup_type_by_patent == "2"].index)

    all_id_strings = df["Patent_ID"].astype(str).unique()
    # arch-suffixed ids never carry their own isApproved (approval lives on
    # the base patent), and multi-arch BASE ids never carry their own topType
    # (it lives on the _archN ids) — both excluded from "missing" checks.
    arch_bases = {p.split("_arch")[0] for p in all_id_strings if "_arch" in p}
    all_ids = {p for p in all_id_strings if "_arch" not in p}

    def _is_disapproved(pid):
        return str(appr_by_patent.get(pid)).strip().lower() == "false"

    new_rows, tt_inherited, appr_inherited, needs_manual = [], [], [], []
    for pid in sorted(all_ids):
        needs_tt = (pd.isna(top_type_by_patent.get(pid))
                    and pid not in arch_bases and not _is_disapproved(pid)
                    and pid not in exact_dup_ids)
        has_appr = pd.notna(appr_by_patent.get(pid))
        if not needs_tt and has_appr:
            continue

        if pid in is_dup_ids:
            root = _resolve_chain_root(pid, dup_target)
            root_tt = top_type_by_patent.get(root)
            root_appr = appr_by_patent.get(root)

            if needs_tt and pd.notna(root_tt):
                new_rows.append({
                    "Patent_ID": pid, "Section": "G1", "Sub_Dimension": "Topology Type",
                    "Field": "topType", "Value": root_tt,
                    "Source": "rule_d_inherited", "Image_Path": None, "pre_process_flags": "",
                })
                tt_inherited.append(pid)
            elif needs_tt:
                needs_manual.append(pid)

            if not has_appr and pd.notna(root_appr):
                appr_mask = (df["Field"] == "isApproved") & (df["Patent_ID"].astype(str) == pid)
                if appr_mask.any():
                    df.loc[appr_mask, "Value"] = root_appr
                else:
                    new_rows.append({
                        "Patent_ID": pid, "Section": "T1", "Sub_Dimension": "T1 — Approval Status",
                        "Field": "isApproved", "Value": root_appr,
                        "Source": "rule_d_inherited", "Image_Path": None, "pre_process_flags": "",
                    })
                appr_inherited.append(pid)
        elif needs_tt:
            # Non-duplicate, approved, single-arch, no topType — needs eyes.
            needs_manual.append(pid)

    if new_rows:
        df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
    if tt_inherited:
        _append_flag(df, tt_inherited, "G1 inherited from duplicate root;")
    if appr_inherited:
        _append_flag(df, appr_inherited, "Approval inherited from duplicate root;")
    if needs_manual:
        _append_flag(df, needs_manual, "Missing topType — needs manual review;")

    print(f"Rule D: topType inherited {len(tt_inherited)} | isApproved inherited "
          f"{len(appr_inherited)} | needs manual review {len(needs_manual)}")
    return df


### Rule E — Ground → Other (per-figure `acState`)

`acState` (Section T2, per figure) has 6 wizard options: `Ground, Hover,
Transition, Cruise, Unclear, NonApplicable`. Verified on Batch_01: `Ground`
is the most ambiguous in practice (a "ground" drawing is often just an
uncommitted static illustration, not a meaningful flight-state observation),
so it's folded into `"Other — Other"`, leaving `Hover`/`Cruise`/`Transition`
as the three real flight states plus `Unclear`/`NonApplicable`/`Other`.

Every `acState` row whose stripped value is `"Ground"` is rewritten to
`"Other — Other"` and its patent flagged `"acState Ground → Other;"`.
`Other` was added as a real option in the HTML wizard
(`UI_for_taxonomy_caracterization_13_0.html`, `AC_STATE` array) so the file
still round-trips if reopened there.


In [10]:
def _remap_ground_to_other(df: pd.DataFrame) -> pd.DataFrame:
    """Rule E — fold per-figure acState "Ground" into "Other"."""
    df = df.copy()

    mask = (df["Field"] == "acState") & (df["Value"].map(_strip_label) == "Ground")
    affected_ids = df.loc[mask, "Patent_ID"].unique().tolist()

    if affected_ids:
        df.loc[mask, "Value"] = "Other — Other"
        _append_flag(df, affected_ids, "acState Ground → Other;")

    print(f"Rule E: {int(mask.sum())} acState row(s) remapped Ground → Other "
          f"({len(affected_ids)} patent(s))")
    return df


### Rule F — Booms X-Formation Backfill (from free-text `boomNotes`)

Already-labelled batches recorded "booms arranged in X formation" as free
text in `boomNotes` (e.g. `US2018354615A1`: *"4 booms arranged in X
formation"*) — there was no structured field for it. The HTML wizard now has
one: a checkbox next to Booms Edge-Case Notes, exported as `Field="boomXFormat"`
(Section `M1`) — captured automatically for every batch labelled from now on.

For batches labelled *before* this checkbox existed, this rule backfills it:
any `boomNotes` row whose text matches `x form(at(ion)?)?` (word-boundary,
case-insensitive — e.g. "X formation", "X-format", or bare "X form") gets a companion
`boomXFormat=True` row added (the original `boomNotes` text is left
untouched). Patents flagged `"Booms X-formation backfilled from notes;"`.


In [11]:
import re

# Matches "X formation", "X-format", "Xform", and bare "X form" (all seen
# in real boomNotes text — e.g. "booms in x form", not just "...formation").
_X_FORMAT_RE = re.compile(r"\bx[\s\-]*form(?:at(?:ion)?)?\b", re.IGNORECASE)


def _backfill_boom_x_format(df: pd.DataFrame) -> pd.DataFrame:
    """Rule F — add boomXFormat=True where boomNotes text mentions X formation
    and no boomXFormat row exists yet (new batches carry it natively).
    """
    df = df.copy()

    already_has = set(df.loc[df["Field"] == "boomXFormat", "Patent_ID"].astype(str))
    notes_rows = df.loc[
        (df["Field"] == "boomNotes")
        & df["Value"].astype(str).str.contains(_X_FORMAT_RE, na=False)
    ]
    backfill_ids = [
        pid for pid in notes_rows["Patent_ID"].astype(str).unique()
        if pid not in already_has
    ]

    if backfill_ids:
        new_rows = pd.DataFrame([
            {
                "Patent_ID": pid, "Section": "M1", "Sub_Dimension": "boomNotes",
                "Field": "boomXFormat", "Value": True,
                "Source": "rule_f_backfill", "Image_Path": None, "pre_process_flags": "",
            }
            for pid in backfill_ids
        ])
        df = pd.concat([df, new_rows], ignore_index=True)
        _append_flag(df, backfill_ids, "Booms X-formation backfilled from notes;")

    print(f"Rule F: {len(backfill_ids)} boomXFormat row(s) backfilled from boomNotes text")
    return df


### Rule G — Ambiguous `fusKin` ("Variable") Flag for Manual Relabel

The current taxonomy (`UI_for_taxonomy_caracterization_13_0.html`, `FUS_KIN`)
splits fuselage kinematics three ways: `Fixed | VarInc | TiltBody`. Some
older-labelled batches (Batch_05: 19 patents) were reviewed while the wizard
had these two merged into one option, `"Variable — Variable Incidence /
Tilting Body"` — not a valid id anymore, and not automatically resolvable
(it genuinely could be either).

Every `fusKin` row whose stripped value is `"Variable"` is flagged
`"fusKin ambiguous (Variable) — needs relabel to VarInc/TiltBody;"` and left
untouched otherwise. The Section 5 UI shows the patent's image and two extra
buttons — **Set: Variable Incidence** / **Set: Tilting Body** — so you can
look at the figure and pick the correct one directly.


In [12]:
def _flag_ambiguous_fuskin(df: pd.DataFrame) -> pd.DataFrame:
    """Rule G — flag patents whose fusKin used the obsolete merged "Variable"
    option (VarInc+TiltBody combined, no longer a valid taxonomy id). Doesn't
    guess which one it should be — the Section 5 UI's Set VarInc/Set
    TiltBody buttons let the reviewer pick after looking at the image.
    """
    df = df.copy()

    mask = (df["Field"] == "fusKin") & (df["Value"].map(_strip_label) == "Variable")
    affected_ids = df.loc[mask, "Patent_ID"].unique().tolist()

    if affected_ids:
        _append_flag(df, affected_ids, "fusKin ambiguous (Variable) — needs relabel to VarInc/TiltBody;")

    print(f"Rule G: {len(affected_ids)} patent(s) flagged for fusKin Variable→VarInc/TiltBody relabel")
    return df


### Completeness Check (report-only — "are all fields filled in?")

Runs last, after Rule D has inherited what it can from duplicate roots. For
every **approved** patent that is not an *exact* duplicate (`duplicateType
== "2"` — the only category that is legitimately label-less; types 1/3/4
carry their own G1-M3), checks that the required label rows exist and are
non-blank: `topType`, `wCount`, `empType`, `latSym`, `longSym`, plus at least
one `_propKin` row somewhere in M3 (every architecture has *some* propulsor
kinematics recorded). Multi-arch patents are checked per `_archN` id.

This doesn't fix anything — it flags gaps as `"Incomplete: <fields>;"` so
they surface in the Section 5 queue, and prints a summary so you can see at
a glance whether a batch is clean before exporting.


In [13]:
_OPTIONAL_FIELD_HINTS = ("note", "comment", "override", "uncertain", "othernote")


def _check_label_completeness(df: pd.DataFrame) -> pd.DataFrame:
    """Report-only: flag approved patents with any blank G1/M1/M2/M3 row
    (excluding free-text/optional fields — notes, quick overrides,
    uncertain-flags, "other" clarifications).

    Only duplicateType "2" ("Images AND aircraft the same" — a literal
    identical copy) is legitimately label-less by design (Section 6 strips
    its own T2/G1/M1-M3 and resolves everything via duplicateId lookup).
    duplicateType "1" (Same Aircraft, wizard auto-copies M1-M3+G1) and "3"
    (Same plane/small changes, copy is an editable starting point) and "4"
    (Component Overlap, fully independent patent) all carry their OWN
    G1-M3 labels and must be checked like any non-duplicate patent —
    exempting every isDuplicate=True patent regardless of type was
    silently hiding genuinely incomplete type-1/3/4 patents from review.
    """
    df = df.copy()

    appr_rows = df.loc[df["Field"] == "isApproved"]
    approved_ids = set(appr_rows.loc[appr_rows["Value"].astype(str) == "True", "Patent_ID"].astype(str))
    duptype_rows = df.loc[df["Field"] == "duplicateType"]
    exact_dup_ids = set(
        duptype_rows.loc[duptype_rows["Value"].map(_strip_label) == "2", "Patent_ID"].astype(str)
    )
    check_ids = approved_ids - exact_dup_ids

    is_optional = df["Field"].astype(str).str.lower().str.contains("|".join(_OPTIONAL_FIELD_HINTS))
    label_rows = df.loc[
        df["Section"].isin(["G1", "M1", "M2", "M3"]) & ~is_optional
        & df["Patent_ID"].astype(str).isin(check_ids)
    ]
    blank_rows = label_rows.loc[label_rows["Value"].isna()]

    incomplete = blank_rows.groupby(blank_rows["Patent_ID"].astype(str))["Field"].apply(
        lambda fields: ", ".join(sorted(set(fields)))
    )

    for pid, fields in incomplete.items():
        _append_flag(df, [pid], f"Incomplete: {fields};")

    print(f"Completeness check: {len(check_ids)} approved non-exact-duplicate patent(s) checked | "
          f"{len(incomplete)} with a blank required field")
    if len(incomplete):
        for pid, fields in incomplete.items():
            print(f"  ⚠ {pid}: {fields}")

    return df


### Orchestration — interactive, per-change confirmation

**Nothing is applied automatically** (unless `HEADLESS=True`, Section 1 — then
every rule is auto-applied to every patent so the notebook can run
unattended). The `RuleReviewPipeline` below walks the
rules one at a time, in order:

0. Ghost-block cleanup → 0b. `empKin`→`empTilts` migration → A. Combined-Thrust
flag → B. Fixed Empennage → C. `UAVSimilar` propagation → D. duplicate
inheritance → E. `acState` Ground→Other → F. `boomXFormat` backfill.

For each rule it does a **dry run**, shows every patent the rule would touch
with a readable diff (`Field: old → new`, `+ row added`, `− row removed`,
`flag: ...`), and gives you a checkbox per patent (all ticked by default):

- **Apply selected → next rule** — only the ticked patents get the rule's
  changes; unticked patents keep their current rows untouched.
- **Skip this rule** — nothing applied, move on.

The pipeline works on its own copy — your `df` is only updated when you run
the `df = pipeline.result()` cell after finishing. The completeness check
(report-only, changes nothing) runs inside that same cell.

Rule A is still flag-only here: applying it just marks the patent for the
Section 5 image UI, where you make the actual TP→CVT call with the drawing
in front of you.


In [14]:
def _drop_ghost_image_blocks(df: pd.DataFrame) -> pd.DataFrame:
    """Step 0 — drop stale duplicate rows within an image block.

    The wizard can leave a stub behind when a figure is re-keyed (same
    Patent_ID + "Image: <file>" Sub_Dimension, same Field appearing twice
    with different values). Blocks are written in save order, so the LAST
    occurrence is the current one — keep it, drop earlier ones.
    """
    df = df.copy()
    is_img = df["Sub_Dimension"].astype(str).str.startswith("Image: ")
    key_cols = ["Patent_ID", "Sub_Dimension", "Field"]
    dup_mask = is_img & df.duplicated(subset=key_cols, keep="last")
    if dup_mask.any():
        ghost_patents = df.loc[dup_mask, "Patent_ID"].unique().tolist()
        df = df.loc[~dup_mask].reset_index(drop=True)
        _append_flag(df, ghost_patents, "Stale image block removed;")
    return df


PIPELINE_RULES = [
    ("0 — Ghost-block cleanup",          _drop_ghost_image_blocks),
    ("0b — empKin → empTilts migration", _convert_empkin_to_emptilts),
    ("A — Combined Thrust (flag for Section 5)", _flag_combined_thrust),
    ("B — Fixed Empennage (empTilts → False)",   _flag_fixed_empennage),
    ("C — UAVSimilar duplicate-chain propagation", _propagate_duplicate_tag),
    ("D — Duplicate inheritance (topType/isApproved)", _inherit_from_duplicate_root),
    ("E — acState Ground → Other",       _remap_ground_to_other),
    ("F — boomXFormat backfill from boomNotes", _backfill_boom_x_format),
    ("G — fusKin 'Variable' ambiguity (flag for Section 5)", _flag_ambiguous_fuskin),
]

_DIFF_COLS = ["Section", "Sub_Dimension", "Field", "Value"]


def _patent_rows(df: pd.DataFrame, pid: str) -> pd.DataFrame:
    return df.loc[df["Patent_ID"].astype(str) == pid]


def _flags_of(df: pd.DataFrame, pid: str) -> set:
    if "pre_process_flags" not in df.columns:
        return set()
    vals = _patent_rows(df, pid)["pre_process_flags"].fillna("").astype(str)
    out = set()
    for v in vals:
        out |= {m.strip() for m in v.split(";") if m.strip()}
    return out


def _affected_patents(before: pd.DataFrame, after: pd.DataFrame) -> list:
    """Patent ids whose rows OR flags differ between before and after."""
    def sig(d):
        cols = [c for c in _DIFF_COLS]
        out = {}
        for pid, g in d.groupby(d["Patent_ID"].astype(str)):
            rows = sorted(map(tuple, g[cols].astype(str).values))
            flags = tuple(sorted(_flags_of(d, pid)))
            out[pid] = (tuple(rows), flags)
        return out
    b, a = sig(before), sig(after)
    return sorted(pid for pid in set(b) | set(a) if b.get(pid) != a.get(pid))


# Human-readable names for the field ids labellers actually see in the widget.
# Falls back to the raw field id (still readable, just less pretty) for
# anything not listed here, so a new field never breaks the description.
_FIELD_LABELS = {
    "topType": "Architecture (topType)", "empTilts": "Empennage Tilts",
    "empKin": "Empennage Kinematics (old)", "figKey": "Figure Number",
    "status": "Image Approval Status", "isMain": "Main Figure",
    "acState": "Flight State", "boomXFormat": "Booms X-Formation",
    "boomNotes": "Booms Notes", "duplicateType": "Duplicate Type",
    "isApproved": "Patent Approved", "isDuplicate": "Is Duplicate",
    "duplicateId": "Duplicate Of (Patent ID)", "t1EdgeTags": "Edge-Case Tags",
    "fusKin": "Fuselage Kinematics", "wCount": "Wing Count",
    "latSym": "Laterally Symmetric", "longSym": "Longitudinally Symmetric",
}


def _field_label(field: str) -> str:
    if field in _FIELD_LABELS:
        return _FIELD_LABELS[field]
    if field.endswith("_propKin"):
        card = field[: -len("_propKin")].replace("_", " ").title()
        return f"{card} — Propulsor Kinematics"
    return field


def _describe_change(before: pd.DataFrame, after: pd.DataFrame, pid: str) -> str:
    """Readable per-patent diff: value changes, added/removed rows, new flags.

    Plain-language field names (see _field_label) instead of raw ids, and
    "removed, nothing added" rows for the SAME image block are grouped into
    one summary sentence (Ghost-block cleanup drops a whole stale duplicate
    block at once — showing each of its fields as a separate cryptic "− ..."
    line reads like data loss when it's really one superseded entry going away).
    """
    from collections import Counter
    b = _patent_rows(before, pid)[_DIFF_COLS].astype(str)
    a = _patent_rows(after, pid)[_DIFF_COLS].astype(str)
    bc, ac = Counter(map(tuple, b.values)), Counter(map(tuple, a.values))
    removed = list((bc - ac).elements())
    added = list((ac - bc).elements())

    msgs = []
    rem_by_key = {}
    for sec, sub, field, val in removed:
        rem_by_key.setdefault((sec, sub, field), []).append(val)
    for sec, sub, field, val in added:
        olds = rem_by_key.get((sec, sub, field))
        if olds:
            msgs.append(f"{_field_label(field)} changed: '{olds.pop(0)}' → '{val}'")
            if not olds:
                del rem_by_key[(sec, sub, field)]
        else:
            msgs.append(f"+ Added {_field_label(field)} = '{val}'")

    # Group whatever's left (removed with no matching add) by (Section, Sub_Dimension).
    leftover_by_block = {}
    for (sec, sub, field), olds in rem_by_key.items():
        for val in olds:
            leftover_by_block.setdefault((sec, sub), []).append((field, val))

    for (sec, sub), field_vals in leftover_by_block.items():
        if sub.startswith("Image: ") and len(field_vals) > 1:
            detail = ", ".join(f"{_field_label(f)}='{v}'" for f, v in field_vals)
            # Same figure file can appear twice in the raw export when a
            # labeller re-keys/re-reviews it later in a separate session —
            # the OLD entry (usually incomplete/disapproved) is what's being
            # removed here. Look up whether a CURRENT entry for the same
            # image still exists in `after`, so the labeller can see at a
            # glance that their later, real review survives untouched.
            surviving = after.loc[
                (after["Patent_ID"].astype(str) == pid)
                & (after["Section"] == sec) & (after["Sub_Dimension"] == sub)
            ]
            surv_status = surviving.loc[surviving["Field"] == "status", "Value"]
            surv_figkey = surviving.loc[surviving["Field"] == "figKey", "Value"]
            if not surviving.empty:
                bits = []
                if len(surv_figkey): bits.append(f"Figure Number='{surv_figkey.iloc[0]}'")
                if len(surv_status): bits.append(f"Image Approval Status='{surv_status.iloc[0]}'")
                survives = f" — this image was re-keyed/re-reviewed later; the CURRENT, KEPT entry for it ({', '.join(bits)}) is untouched." if bits else \
                           " — this image was re-keyed/re-reviewed later; a current entry for it is untouched."
            else:
                survives = " (no current entry for this image survives — check it wasn't accidentally lost)"
            msgs.append(f"Removed an OLD, superseded entry for {sub} (it had {detail}){survives}")
        else:
            for field, val in field_vals:
                msgs.append(f"− Removed {_field_label(field)} (was '{val}')")

    for f in sorted(_flags_of(after, pid) - _flags_of(before, pid)):
        msgs.append(f'Flag added: "{f.rstrip(";")}"')
    return "; ".join(msgs) if msgs else "(no visible change)"


def _merge_selected(before: pd.DataFrame, after: pd.DataFrame, selected: set) -> pd.DataFrame:
    """Take `after`'s rows for selected patents, `before`'s for everyone else,
    restoring the original patent order (rules only ever touch a patent's own
    rows, so per-patent row swapping is safe)."""
    for d in (before, after):
        if "pre_process_flags" not in d.columns:
            d["pre_process_flags"] = ""
    b_pid = before["Patent_ID"].astype(str)
    a_pid = after["Patent_ID"].astype(str)
    merged = pd.concat([before[~b_pid.isin(selected)], after[a_pid.isin(selected)]],
                       ignore_index=True)
    order = {pid: i for i, pid in enumerate(dict.fromkeys(b_pid))}
    merged["_o"] = merged["Patent_ID"].astype(str).map(lambda p: order.get(p, len(order)))
    merged = merged.sort_values("_o", kind="stable").drop(columns="_o").reset_index(drop=True)
    return merged



# ── thumbnails for the confirmation rows ─────────────────────────────────────
from io import BytesIO
from PIL import Image as _PILImage

_PIPELINE_MATCHED_DIR = Path(cfg["paths"]["matched"]) / sheet_name
_THUMB_CACHE: dict = {}


def _t2_image_for_patent(df: pd.DataFrame, patent_id: str):
    """Best (Image_Path, label) for THIS architecture's OWN reviewed T2 data.

    Multi-architecture patents keep every T2 (figure) row on the BASE
    patent id only — "_arch2"/"_arch3" ids carry no T2 rows of their own —
    but each figure block IS tagged with which architecture it belongs to
    (Field="arch", values 1..N, matching parse_arch_id()'s arch number).
    Filters to the figures actually tagged for this architecture first.

    Returns (None, None) if this patent has no T2 rows at all (e.g. an
    exact-duplicate patent, which by design was never independently
    reviewed) — callers must NOT silently show a random disk file in that
    case without labelling it as unreviewed (see _resolve_patent_image).
    """
    base_id, arch_num = proc.parse_arch_id(patent_id)
    t2 = df.loc[(df["Patent_ID"].astype(str) == base_id) & (df["Section"] == "T2")
                & df["Image_Path"].notna()]
    if t2.empty:
        return None, None

    arch_tag_rows = df.loc[(df["Patent_ID"].astype(str) == base_id) & (df["Field"] == "arch")]
    arch_by_block = arch_tag_rows.set_index("Sub_Dimension")["Value"]

    def _arch_of(sub):
        v = arch_by_block.get(sub)
        try:
            return int(_strip_label(v)) if v is not None and pd.notna(v) else None
        except (TypeError, ValueError):
            return None

    t2 = t2.copy()
    t2["_arch_tag"] = t2["Sub_Dimension"].map(_arch_of)
    if t2["_arch_tag"].notna().any():
        this_arch = t2[t2["_arch_tag"] == arch_num]
        if not this_arch.empty:
            t2 = this_arch  # else: nothing tagged for this arch — fall back to all figures below

    is_main = t2.loc[(t2["Field"] == "isMain")
                     & t2["Value"].astype(str).str.lower().isin(["true", "1", "yes", "main"]),
                     "Image_Path"]
    if not is_main.empty:
        return Path(str(is_main.iloc[0])), "✅ Approved main figure"
    any_path = t2["Image_Path"].dropna()
    if not any_path.empty:
        return Path(str(any_path.iloc[0])), "Reviewed figure (not marked main)"
    return None, None


def _resolve_patent_image(df: pd.DataFrame, pid: str, matched_dir_root: Path):
    """(path_or_None, provenance_label) for the thumbnail/preview shown to
    the reviewer — never silently passes off an unreviewed raw file as if
    it were an approved figure.

    Order: (1) this patent's own reviewed T2 data; (2) if it's a duplicate
    with no T2 of its own (the normal case for duplicateType "2" — "Images
    AND aircraft the same" — which is never independently reviewed by
    design), follow duplicateId to the chain root and show THAT patent's
    approved figure instead, clearly labelled as inherited; (3) only as a
    last resort, an arbitrary raw file from disk, clearly labelled as
    unreviewed so it can't be mistaken for an approved main image.
    """
    path, label = _t2_image_for_patent(df, pid)
    if path is not None:
        return path, label

    dup_rows = df.loc[(df["Patent_ID"].astype(str) == pid) & (df["Field"] == "duplicateId")
                       & df["Value"].notna() & (df["Value"].astype(str).str.strip() != "")]
    if not dup_rows.empty:
        dup_target = df.loc[(df["Field"] == "duplicateId") & df["Value"].notna()]
        dup_target = dup_target.set_index(dup_target["Patent_ID"].astype(str))["Value"]
        root = _resolve_chain_root(pid, dup_target)
        if root != pid:
            root_path, root_label = _t2_image_for_patent(df, root)
            if root_path is not None:
                return root_path, f"⚠ No figure of its own (duplicate) — showing ORIGINAL {root}'s {root_label.lower()}"

    base = pid.split("_arch")[0]
    try:
        hits = sorted(matched_dir_root.glob(f"{base}_*/*"))
    except OSError:
        hits = []
    if hits:
        return hits[0], "⚠ UNREVIEWED raw file from disk — NOT an approved figure"
    return None, None


def _patent_thumbnail(df: pd.DataFrame, pid: str, max_px: int = 480):
    """(PNG bytes or None, provenance label or None) preview for pid, via
    _resolve_patent_image — labelled so an unreviewed/inherited fallback is
    never mistaken for an approved figure. max_px=480 sets the LONG edge;
    the short edge is whatever the image's real aspect ratio gives it (a
    fixed width previously squashed wide landscape crops into thin strips)."""
    if pid in _THUMB_CACHE:
        return _THUMB_CACHE[pid]
    path, label = _resolve_patent_image(df, pid, _PIPELINE_MATCHED_DIR)
    data = None
    if path is not None and path.exists():
        try:
            im = _PILImage.open(path); im.thumbnail((max_px, max_px))
            buf = BytesIO(); im.convert("RGB").save(buf, "PNG")
            data = buf.getvalue()
        except Exception:
            data = None
    _THUMB_CACHE[pid] = (data, label)
    return data, label


class RuleReviewPipeline:
    """Step through PIPELINE_RULES; every proposed change needs your tick."""

    def __init__(self, df: pd.DataFrame, rules=None, auto_apply: bool = False):
        self.df = df.copy()
        self.rules = list(rules or PIPELINE_RULES)
        self.step = 0
        self.finished = False
        self.auto_apply = auto_apply   # True → apply every rule to every patent, no widget
        self._after = None
        self._boxes = {}

        self.title = widgets.HTML()
        self.list_box = widgets.VBox(
            layout=widgets.Layout(max_height="900px", overflow_y="auto",
                                  border="1px solid #ccc", padding="6px"))
        self.all_btn = widgets.Button(description="Select all")
        self.none_btn = widgets.Button(description="Select none")
        self.apply_btn = widgets.Button(description="Apply selected → next rule",
                                        button_style="success")
        self.skip_btn = widgets.Button(description="Skip this rule (apply nothing)",
                                       button_style="warning")
        self.log = widgets.HTML()
        self.all_btn.on_click(lambda b: self._set_all(True))
        self.none_btn.on_click(lambda b: self._set_all(False))
        self.apply_btn.on_click(self._on_apply)
        self.skip_btn.on_click(self._on_skip)
        self.root = widgets.VBox([self.title,
                                  widgets.HBox([self.all_btn, self.none_btn]),
                                  self.list_box,
                                  widgets.HBox([self.apply_btn, self.skip_btn]),
                                  self.log])
        self._render_step()

    def _set_all(self, value):
        for box in self._boxes.values():
            box.value = value

    def _render_step(self):
        while self.step < len(self.rules):
            name, fn = self.rules[self.step]
            self._after = fn(self.df)
            affected = _affected_patents(self.df, self._after)
            if not affected:
                self.log.value += f"<div>• <b>{name}</b>: nothing to change — auto-skipped.</div>"
                self.step += 1
                continue
            if self.auto_apply:
                self.df = _merge_selected(self.df, self._after, set(affected))
                self.log.value += (f"<div>• <b>{name}</b>: AUTO-applied to "
                                   f"{len(affected)} patent(s) (headless).</div>")
                print(f"  [headless] {name}: auto-applied to {len(affected)} patent(s)")
                self.step += 1
                continue
            self.title.value = (f"<h4>Rule {name}</h4><b>{len(affected)}</b> patent(s) "
                                f"would change — untick anything you do NOT want applied "
                                f"(step {self.step + 1}/{len(self.rules)}):")
            self._boxes = {}
            # One INDEPENDENT HBox per patent (not a shared GridBox) — each
            # row lays itself out with no cross-row height-sharing, which is
            # what broke: GridBox's row height came from its tallest cell,
            # but different figures have very different aspect ratios, so
            # rows ended up misaligned between the image and text columns.
            # widgets.Image (not raw HTML <img>, which VSCode collapsed to
            # scrollbar bars) + explicit flex-basis per column (the fix that
            # stopped the image container from stretching past its image).
            items = []
            for pid in affected:
                box = widgets.Checkbox(value=True, indent=False,
                                       layout=widgets.Layout(width="28px", flex="0 0 28px"))
                self._boxes[pid] = box
                desc = _describe_change(self.df, self._after, pid)
                thumb, thumb_label = _patent_thumbnail(self.df, pid)
                if thumb:
                    img = widgets.Image(value=thumb, format="png",
                                        layout=widgets.Layout(max_width="420px",
                                                              max_height="420px"))
                    warn = bool(thumb_label) and thumb_label.startswith("⚠")
                    cap = widgets.HTML(
                        f"<span style='font-size:11px;color:{'#b45309' if warn else '#555'}'>"
                        f"{thumb_label or ''}</span>")
                    img_col = widgets.VBox([img, cap],
                                           layout=widgets.Layout(flex="0 0 440px",
                                                                 align_items="flex-start"))
                else:
                    img_col = widgets.HTML("<i style='color:#999'>no image</i>",
                                           layout=widgets.Layout(flex="0 0 440px"))
                text_w = widgets.HTML(f"<b>{pid}</b> — {desc}",
                                      layout=widgets.Layout(margin="0 0 0 12px",
                                                            min_width="300px", flex="1 1 auto"))
                items.append(widgets.HBox(
                    [box, img_col, text_w],
                    layout=widgets.Layout(align_items="flex-start",
                                          border_bottom="1px solid #eee",
                                          padding="10px 0", width="100%")))
            self.list_box.children = items
            return
        self._finish()

    def _on_apply(self, _btn):
        name, _ = self.rules[self.step]
        selected = {pid for pid, box in self._boxes.items() if box.value}
        self.df = _merge_selected(self.df, self._after, selected)
        n_rej = len(self._boxes) - len(selected)
        self.log.value += (f"<div>• <b>{name}</b>: applied to {len(selected)} patent(s)"
                           + (f", rejected {n_rej}" if n_rej else "") + ".</div>")
        self.step += 1
        self._render_step()

    def _on_skip(self, _btn):
        name, _ = self.rules[self.step]
        self.log.value += f"<div>• <b>{name}</b>: skipped (nothing applied).</div>"
        self.step += 1
        self._render_step()

    def _finish(self):
        self.finished = True
        self.title.value = "<h4>✅ All rules reviewed.</h4>Run the next cell: <code>df = pipeline.result()</code>"
        self.list_box.children = []
        self.apply_btn.disabled = self.skip_btn.disabled = True
        self.all_btn.disabled = self.none_btn.disabled = True

    def result(self) -> pd.DataFrame:
        assert self.finished, "Finish reviewing all rules in the widget first."
        return self.df.copy()

    def display(self):
        display(self.root)


pipeline = RuleReviewPipeline(df, auto_apply=HEADLESS)
if HEADLESS:
    print("HEADLESS: all rules auto-applied above — no widget to click.")
else:
    pipeline.display()


In [16]:
# Run AFTER finishing the widget above — commits your confirmed changes to df
# and runs the report-only completeness check (flags gaps, changes no labels).
df = pipeline.result()
df = _check_label_completeness(df)

# Per-flag patent counts — the queue Section 5 will walk.
_per_flag, _flagged_patents = {}, 0
for pid, g in df.groupby(df["Patent_ID"].astype(str)):
    flags = {m.strip() for v in g["pre_process_flags"].fillna("").astype(str)
             for m in v.split(";") if m.strip()}
    if flags:
        _flagged_patents += 1
    for f in flags:
        _per_flag[f] = _per_flag.get(f, 0) + 1
print(f"{_flagged_patents} patent(s) carry at least one flag:")
for f, n in sorted(_per_flag.items(), key=lambda kv: -kv[1]):
    print(f"  {n:4d}  {f}")


Completeness check: 125 approved non-exact-duplicate patent(s) checked | 0 with a blank required field
87 patent(s) carry at least one flag:
    32  Approval inherited from duplicate root
    30  acState Ground → Other
    15  UAV Tag Propagated
     8  Review: Potential Combined Thrust (TP→CVT?)
     3  Booms X-formation backfilled from notes
     1  Stale image block removed
     1  G1 inherited from duplicate root
     1  Empennage tilts (empKin migrated to empTilts)


## Section 5 — Interactive Review UI

One flagged patent at a time. Left pane renders its main architecture image
from `matched/<batch>/<patent_id>_*/`; right pane shows Patent ID, Company,
Architecture (both from `data/batches.xlsx`, keyed on the base patent id —
see `src/grouper.py`), and the `pre_process_flags` from Section 4.

- **Update to Combined Thrust** / **Keep As Is** both record the reviewer's
  decision into `pre_process_flags` and advance to the next flagged record;
  the former also (re)writes `topType` to `"Combined Thrust"`.
- **Prev** / **Next** move through the queue without changing anything.
- The search box filters the queue by substring match against Company,
  Architecture, or Patent ID (case-insensitive).
- A missing/unreadable image never raises — the image pane shows a
  short warning instead.


In [17]:
matched_dir = Path(cfg["paths"]["matched"]) / sheet_name


def build_issue_queue(df: pd.DataFrame, cfg: dict, sheet_name: str) -> pd.DataFrame:
    """One row per flagged Patent_ID: flags text, current topType, and
    Company/Architecture pulled from data/batches.xlsx (keyed on the base,
    non-arch-suffixed patent id — see src/grouper.py).
    """
    flags_col = df.get("pre_process_flags")
    if flags_col is None:
        return pd.DataFrame(columns=["Patent_ID", "flags", "topType", "Company", "Architecture"])

    flagged_ids = df.loc[flags_col.astype(str).str.len() > 0, "Patent_ID"].unique()
    queue = pd.DataFrame({"Patent_ID": flagged_ids})

    flags_by_patent = (
        df[df["Patent_ID"].isin(flagged_ids)]
        .groupby("Patent_ID")["pre_process_flags"]
        .apply(lambda s: max(s.dropna(), key=len, default=""))
    )
    top_type_by_patent = df.loc[df["Field"] == "topType"].set_index("Patent_ID")["Value"]
    queue["flags"] = queue["Patent_ID"].map(flags_by_patent)
    queue["topType"] = queue["Patent_ID"].map(top_type_by_patent)

    base_ids = queue["Patent_ID"].map(lambda pid: proc.parse_arch_id(pid)[0])
    try:
        batches_df = pd.read_excel(cfg["paths"]["batches_xlsx"], sheet_name=sheet_name, dtype=str)
        meta = batches_df.drop_duplicates("patent_id").set_index("patent_id")
        queue["Company"] = base_ids.map(meta.get("company_canonical", pd.Series(dtype=str)))
        queue["Architecture"] = base_ids.map(meta.get("prototype_label", pd.Series(dtype=str)))
    except (FileNotFoundError, ValueError, KeyError):
        # batches.xlsx missing/unreadable for this sheet — search still works
        # against Patent_ID alone.
        queue["Company"] = None
        queue["Architecture"] = None

    return queue.reset_index(drop=True)


def load_main_architecture_image(df: pd.DataFrame, patent_id: str, matched_dir: Path):
    """(path_or_None, provenance_label) for patent_id via _resolve_patent_image
    — see that function for the approved / inherited-from-duplicate-root /
    unreviewed-raw-file distinction. Never raises."""
    return _resolve_patent_image(df, patent_id, matched_dir)


class ReviewSession:
    """One-issue-at-a-time review widget over an issue_queue DataFrame."""

    def __init__(self, df: pd.DataFrame, issue_queue: pd.DataFrame, matched_dir: Path):
        self.df = df                 # mutated in place by the action buttons
        self.full_queue = issue_queue
        self.queue = issue_queue
        self.matched_dir = matched_dir
        self.pos = 0

        self.search_box = widgets.Text(
            description="Search:",
            placeholder="Filter by Company / Architecture / Patent ID / flag reason "
                        "(e.g. \"Combined Thrust\", \"fusKin\", \"UAV\")",
            layout=widgets.Layout(width="620px"),
        )
        self.search_box.observe(self._on_search, names="value")

        self.status_label = widgets.Label()
        self.image_panel = widgets.Output(layout=widgets.Layout(width="45%", border="1px solid #ccc"))
        self.meta_panel = widgets.HTML(layout=widgets.Layout(width="55%"))

        self.prev_btn = widgets.Button(description="◀ Prev")
        self.next_btn = widgets.Button(description="Next ▶")
        self.cvt_btn = widgets.Button(description="Update to CVT (Combined)", button_style="warning")
        self.varinc_btn = widgets.Button(description="fusKin: Variable Incidence", button_style="info")
        self.tiltbody_btn = widgets.Button(description="fusKin: Tilting Body", button_style="info")
        self.keep_btn = widgets.Button(description="Keep As Is", button_style="success")

        self.prev_btn.on_click(self._on_prev)
        self.next_btn.on_click(self._on_next)
        self.cvt_btn.on_click(self._on_cvt)
        self.varinc_btn.on_click(self._on_set_varinc)
        self.tiltbody_btn.on_click(self._on_set_tiltbody)
        self.keep_btn.on_click(self._on_keep)

        self.action_box = widgets.HBox([self.cvt_btn, self.varinc_btn, self.tiltbody_btn])
        self.root = widgets.VBox([
            widgets.HBox([self.search_box]),
            self.status_label,
            widgets.HBox([self.image_panel, self.meta_panel]),
            widgets.HBox([self.prev_btn, self.next_btn, self.action_box, self.keep_btn]),
        ])
        self._render()

    # ── navigation / filtering ──────────────────────────────────────────
    def _on_search(self, change):
        term = (change["new"] or "").strip().lower()
        if not term:
            self.queue = self.full_queue
        else:
            mask = self.full_queue.apply(
                lambda r: term in str(r.get("Company", "")).lower()
                or term in str(r.get("Architecture", "")).lower()
                or term in str(r["Patent_ID"]).lower()
                or term in str(r.get("flags", "")).lower(),
                axis=1,
            )
            self.queue = self.full_queue[mask].reset_index(drop=True)
        self.pos = 0
        self._render()

    def _on_prev(self, _btn):
        if self.pos > 0:
            self.pos -= 1
            self._render()

    def _on_next(self, _btn):
        if self.pos < len(self.queue) - 1:
            self.pos += 1
            self._render()

    def _current_patent_id(self):
        return None if self.queue.empty else self.queue.iloc[self.pos]["Patent_ID"]

    # ── decisions ────────────────────────────────────────────────────────
    def _on_cvt(self, _btn):
        """Reviewer confirms Combined Thrust: topType TP → CVT (the wizard's
        own composite value, so the file stays round-trip compatible)."""
        pid = self._current_patent_id()
        if pid is not None:
            mask = (self.df["Field"] == "topType") & (self.df["Patent_ID"] == pid)
            self.df.loc[mask, "Value"] = CVT_VALUE
            _append_flag(self.df, [pid], "Reviewer: Confirmed CVT (Combined Thrust);")
        self._advance()

    def _set_fuskin(self, value: str, flag_msg: str):
        """Rule G resolution: reviewer picks the correct fusKin after
        looking at the image (only acts if the patent actually has a
        fusKin row — a no-op button press otherwise)."""
        pid = self._current_patent_id()
        if pid is not None:
            mask = (self.df["Field"] == "fusKin") & (self.df["Patent_ID"] == pid)
            if mask.any():
                self.df.loc[mask, "Value"] = value
                _append_flag(self.df, [pid], flag_msg)
        self._advance()

    def _on_set_varinc(self, _btn):
        self._set_fuskin("VarInc — Variable Incidence", "Reviewer: fusKin set to Variable Incidence;")

    def _on_set_tiltbody(self, _btn):
        self._set_fuskin("TiltBody — Tilting Body", "Reviewer: fusKin set to Tilting Body;")

    def _on_keep(self, _btn):
        pid = self._current_patent_id()
        if pid is not None:
            _append_flag(self.df, [pid], "Reviewer: Kept As Is;")
        self._advance()

    def _advance(self):
        if self.pos < len(self.queue) - 1:
            self.pos += 1
            self._render()
        else:
            self._render()
            self.status_label.value = (
                f"✅ Reviewed {len(self.queue)} of {len(self.queue)} — end of queue. "
                f"Continue to Section 5b / Section 6.")

    # ── rendering ────────────────────────────────────────────────────────
    def _render(self):
        n = len(self.queue)
        if n == 0:
            self.status_label.value = "No flagged records match this filter."
            self.meta_panel.value = ""
            with self.image_panel:
                clear_output(wait=True)
            return

        self.pos = min(self.pos, n - 1)
        row = self.queue.iloc[self.pos]
        self.status_label.value = f"Reviewing {self.pos + 1} of {n}"
        flags_text = str(row.get("flags") or "")
        self.meta_panel.value = (
            f"<b>Patent ID:</b> {row['Patent_ID']}<br>"
            f"<b>Company:</b> {row.get('Company') or '—'}<br>"
            f"<b>Architecture:</b> {row.get('Architecture') or '—'}<br>"
            f"<b>topType:</b> {row.get('topType') or '—'}<br>"
            f"<b>Flags:</b> {flags_text or '—'}"
        )

        # Only show the button(s) that actually apply to THIS patent's flag —
        # all 4 buttons used to always show regardless of which rule fired,
        # so it was unclear which one was relevant for the patent on screen.
        is_cvt_case = "Combined Thrust" in flags_text
        is_fuskin_case = "fusKin ambiguous" in flags_text
        self.cvt_btn.layout.display = "" if is_cvt_case else "none"
        self.varinc_btn.layout.display = "" if is_fuskin_case else "none"
        self.tiltbody_btn.layout.display = "" if is_fuskin_case else "none"

        with self.image_panel:
            clear_output(wait=True)
            try:
                image_path, image_label = load_main_architecture_image(self.df, row["Patent_ID"], self.matched_dir)
            except Exception as exc:  # never let a lookup error break the UI
                print(f"⚠ image lookup failed for {row['Patent_ID']}: {exc}")
                image_path, image_label = None, None

            if image_path is None:
                print(f"⚠ no image found for {row['Patent_ID']}")
            else:
                if image_label and image_label.startswith("⚠"):
                    print(image_label)  # unreviewed/inherited fallback — flag it BEFORE the image
                else:
                    print(image_label or "")
                try:
                    display(widgets.Image(
                        value=image_path.read_bytes(),
                        format=image_path.suffix.lstrip(".") or "png",
                        layout=widgets.Layout(max_width="100%"),
                    ))
                except Exception as exc:
                    print(f"⚠ could not load image {image_path}: {exc}")

    def display(self):
        display(self.root)


def build_review_grid(df: pd.DataFrame, issue_queue: pd.DataFrame, matched_dir: Path = matched_dir) -> ReviewSession:
    session = ReviewSession(df, issue_queue, matched_dir)
    session.display()
    return session


issue_queue = build_issue_queue(df, cfg, sheet_name)
print(f"Section 5 queue: {len(issue_queue)} flagged patent(s)")
if not issue_queue.empty:
    _per_flag_q = {}
    for flags_text in issue_queue["flags"].fillna(""):
        for f in {m.strip() for m in str(flags_text).split(";") if m.strip()}:
            _per_flag_q[f] = _per_flag_q.get(f, 0) + 1
    print("Breakdown (search box below matches any of these — try typing part of one):")
    for f, n in sorted(_per_flag_q.items(), key=lambda kv: -kv[1]):
        print(f"  {n:4d}  {f}")
if HEADLESS:
    print("HEADLESS: skipping the interactive review UI (queue above is informational).")
elif issue_queue.empty:
    print("Nothing flagged — skip straight to Section 5b / Section 6.")
else:
    session = build_review_grid(df, issue_queue)


Section 5 queue: 87 flagged patent(s)
Breakdown (search box below matches any of these — try typing part of one):
    32  Approval inherited from duplicate root
    30  acState Ground → Other
    15  UAV Tag Propagated
     8  Review: Potential Combined Thrust (TP→CVT?)
     3  Booms X-formation backfilled from notes
     1  Stale image block removed
     1  Empennage tilts (empKin migrated to empTilts)
     1  G1 inherited from duplicate root


## Section 5b — Missing Main-Figure Review

Some patents reach this stage with **zero reviewed T2 images** — every
`status`/`isMain` row is blank because the reviewer never opened the image
step for them in the wizard (found via a completeness check against the
approved-figure count, not a hardcoded list — currently `US2018354616A1` in
Batch_01; `KR102760903B1`, `US2022089276A1`, `US2022348339A1`,
`WO2023272353A1` in Batch_05). Without at least one `status="approved"` +
`isMain=True` image, these patents can't feed the embeddings pipeline.

This section queues exactly those patents (any patent with T2 rows but no
`isMain=True` among them) and lets you, per image: mark **Approved** and
pick the one **Main** figure. **Save & Next** writes the decisions into
`df`'s `status`/`isMain` rows in place (the same in-memory `df` Section 4
operates on) and appends a `pre_process_flags` note — nothing touches disk
until Section 6's export.

In [18]:
def find_missing_main_figure_patents(df: pd.DataFrame) -> list:
    """Patents with T2 image rows but no isMain=True among them.

    Distinct from duplicates (which correctly carry zero T2 rows at all —
    see Rule D above) — this only catches patents whose image-review step
    was itself skipped or left incomplete.
    """
    t2 = df.loc[df["Section"] == "T2"]
    has_t2 = set(t2["Patent_ID"].unique())
    is_main_true = t2.loc[
        (t2["Field"] == "isMain") & (t2["Value"].astype(str).str.lower() == "true"),
        "Patent_ID",
    ]
    return sorted(has_t2 - set(is_main_true.unique()))


class MainFigureSession:
    """One patent at a time: review every T2 image, mark it Approved/not,
    and pick exactly one Main figure. Writes directly into `self.df`.
    """

    def __init__(self, df: pd.DataFrame, patent_ids: list, matched_dir_root: Path):
        self.df = df  # mutated in place by Save & Next
        self.patent_ids = patent_ids
        self.matched_dir_root = matched_dir_root
        self.pos = 0

        self.status_label = widgets.Label()
        self.images_panel = widgets.VBox()
        self.save_btn = widgets.Button(description="Save & Next", button_style="success")
        self.skip_btn = widgets.Button(description="Skip (no change)")
        self.prev_btn = widgets.Button(description="◀ Prev")

        self.save_btn.on_click(self._on_save)
        self.skip_btn.on_click(self._on_skip)
        self.prev_btn.on_click(self._on_prev)

        self.root = widgets.VBox([
            self.status_label,
            self.images_panel,
            widgets.HBox([self.prev_btn, self.skip_btn, self.save_btn]),
        ])
        self._render()

    def _current_patent_id(self):
        return None if not self.patent_ids else self.patent_ids[self.pos]

    def _render(self):
        n = len(self.patent_ids)
        if n == 0:
            self.status_label.value = "No patents need a main-figure review."
            self.images_panel.children = []
            return

        pid = self._current_patent_id()
        self.status_label.value = f"Reviewing {self.pos + 1} of {n} — {pid}"

        t2 = self.df.loc[(self.df["Patent_ID"] == pid) & (self.df["Section"] == "T2")]
        sub_dims = sorted(t2["Sub_Dimension"].dropna().unique())

        main_group = widgets.RadioButtons(options=["(none)"] + sub_dims, value="(none)",
                                           description="Main:", layout=widgets.Layout(width="100%"))
        approve_boxes = {}
        rows = []
        for sub_dim in sub_dims:
            img_rows = t2.loc[t2["Sub_Dimension"] == sub_dim]
            image_path_str = (img_rows["Image_Path"].dropna().iloc[0]
                               if img_rows["Image_Path"].notna().any() else None)
            out = widgets.Output(layout=widgets.Layout(width="220px", border="1px solid #ccc"))
            with out:
                if image_path_str and Path(image_path_str).exists():
                    p = Path(image_path_str)
                    display(widgets.Image(value=p.read_bytes(), format=p.suffix.lstrip(".") or "png",
                                           layout=widgets.Layout(max_width="200px")))
                else:
                    print(f"⚠ no image at {image_path_str}")
            approve_box = widgets.Checkbox(value=False, description="Approved", indent=False)
            approve_boxes[sub_dim] = approve_box
            rows.append(widgets.VBox([out, widgets.Label(sub_dim), approve_box],
                                      layout=widgets.Layout(margin="4px")))

        self._approve_boxes = approve_boxes
        self._main_group = main_group

        grid = widgets.HBox(rows, layout=widgets.Layout(flex_flow="row wrap"))
        self.images_panel.children = [grid, main_group]

    def _write_decisions(self):
        pid = self._current_patent_id()
        if pid is None:
            return
        main_choice = self._main_group.value
        for sub_dim, approve_box in self._approve_boxes.items():
            status_mask = (
                (self.df["Patent_ID"] == pid)
                & (self.df["Sub_Dimension"] == sub_dim)
                & (self.df["Field"] == "status")
            )
            self.df.loc[status_mask, "Value"] = "approved" if approve_box.value else "disapproved"

            is_main_mask = (
                (self.df["Patent_ID"] == pid)
                & (self.df["Sub_Dimension"] == sub_dim)
                & (self.df["Field"] == "isMain")
            )
            self.df.loc[is_main_mask, "Value"] = (sub_dim == main_choice)

        _append_flag(self.df, [pid], "Main figure reviewed in 02a;")

    def _on_save(self, _btn):
        self._write_decisions()
        self._advance()

    def _on_skip(self, _btn):
        self._advance()

    def _on_prev(self, _btn):
        if self.pos > 0:
            self.pos -= 1
            self._render()

    def _advance(self):
        if self.pos < len(self.patent_ids) - 1:
            self.pos += 1
            self._render()
        else:
            self.status_label.value = "Done — no more patents in the queue."
            self.images_panel.children = []

    def display(self):
        display(self.root)


def build_main_figure_review(df: pd.DataFrame, matched_dir_root: Path = Path(cfg["paths"]["matched"])) -> MainFigureSession:
    patent_ids = find_missing_main_figure_patents(df)
    print(f"{len(patent_ids)} patent(s) need a main-figure review: {patent_ids}")
    session = MainFigureSession(df, patent_ids, matched_dir_root)
    session.display()
    return session


if HEADLESS:
    _missing_main = find_missing_main_figure_patents(df)
    print(f"HEADLESS: {len(_missing_main)} patent(s) would need a main-figure "
          f"review: {_missing_main} (UI skipped)")
else:
    main_figure_session = build_main_figure_review(df)


1 patent(s) need a main-figure review: ['US2018354616A1']


## Section 6 — Export (approved images only)

Writes `Review_postprocess_<batch>_<timestamp>.xlsx` **next to the raw
export** in `reviewed xlsxs/` (config `html_review_exports`), via
`format_review_workbook` (flat `Review` sheet for 02b + merged `Compact`
sheet for humans). The raw `reviewed_patents_<batch>.xlsx` is never touched.

Two scope rules are enforced before the T2-approval filter runs (belt-and-
suspenders with what the wizard itself already omits on export, and with
Rule D's `_inherit_from_duplicate_root` no longer back-filling these):

- **Disapproved patents** (`isApproved == False`) keep only their `T1` rows
  — no G1/M1–M3/T2/META noise from an unreviewed patent.
- **Exact duplicates** (`duplicateType == "2"`, "Images AND aircraft the
  same") keep no `T2`/`G1`/`M1`/`M2`/`M3` rows of their own — the pipeline
  resolves their full labels via `duplicateId` lookup on the original patent.
  "Same Aircraft" duplicates (`duplicateType == "1"`) are unaffected: they
  legitimately carry their own `G1`–`M3` (copied from the original, editable).

After that, T2 image blocks are filtered to **status = approved only** —
disapproved figures, `(fig N)` placeholders and abandoned clipboard pastes
are dropped.


In [19]:
# ── Enforce per-patent row scope ─────────────────────────────────────────────
# Disapproved patents: T1 only. Exact duplicates (duplicateType "2"): no
# T2/G1/M1-M3 of their own (labels come from a duplicateId lookup on the
# original patent — see the Section 6 markdown above and Rule D's
# exact_dup_ids exclusion). "Same Aircraft" duplicates (duplicateType "1")
# keep their own G1-M3 untouched.
appr_rows = df.loc[df["Field"] == "isApproved"]
disapproved_ids = set(
    appr_rows.loc[appr_rows["Value"].astype(str).str.strip().str.lower() == "false", "Patent_ID"].astype(str)
)

dup_type_rows = df.loc[df["Field"] == "duplicateType"]
exact_dup_ids = set(
    dup_type_rows.loc[dup_type_rows["Value"].map(_strip_label) == "2", "Patent_ID"].astype(str)
)

pid_str = df["Patent_ID"].astype(str)
drop_non_t1 = pid_str.isin(disapproved_ids) & (df["Section"] != "T1")
drop_exact_dup_labels = pid_str.isin(exact_dup_ids) & df["Section"].isin(["T2", "G1", "M1", "M2", "M3"])

df = df[~(drop_non_t1 | drop_exact_dup_labels)].reset_index(drop=True)
print(f"scope enforcement: dropped {int(drop_non_t1.sum())} non-T1 row(s) for "
      f"{len(disapproved_ids)} disapproved patent(s) | dropped "
      f"{int(drop_exact_dup_labels.sum())} T2/G1/M1-M3 row(s) for "
      f"{len(exact_dup_ids)} exact-duplicate patent(s)")

# ── Keep only APPROVED image blocks ─────────────────────────────────────────
# A T2 block = all rows sharing (Patent_ID, "Image: <name>" Sub_Dimension).
# Keep a block only if its status row says "approved"; every remaining
# non-T2 row (T1/G1/M1–M3/META labels) is kept unchanged.
t2_status = df[(df["Section"] == "T2") & (df["Field"] == "status")]
approved_blocks = {
    (str(r["Patent_ID"]), str(r["Sub_Dimension"]))
    for _, r in t2_status.iterrows()
    if str(r["Value"]).strip().lower() == "approved"
}
is_t2 = df["Section"] == "T2"
block_key = list(zip(df["Patent_ID"].astype(str), df["Sub_Dimension"].astype(str)))
keep = ~is_t2 | pd.Series([k in approved_blocks for k in block_key], index=df.index)

export_df = df[keep].reset_index(drop=True)
n_dropped_blocks = t2_status.shape[0] - len(approved_blocks)
print(f"approved-only filter: kept {len(export_df)}/{len(df)} rows "
      f"({len(approved_blocks)} approved image blocks, {n_dropped_blocks} blocks dropped)")

# ── Write the new batch file (never overwrites anything) ────────────────────
OUTPUT_DIR = Path(cfg["paths"]["html_review_exports"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_XLSX = OUTPUT_DIR / f"Review_postprocess_{sheet_name}_{timestamp}.xlsx"

assert OUTPUT_XLSX != REVIEWED_XLSX, "refusing to overwrite the raw wizard export"
assert not OUTPUT_XLSX.exists(), f"{OUTPUT_XLSX} already exists — refusing to overwrite"

# ── Sanity assertions (fail loudly BEFORE handing the file to 02b) ─────────
_pid = export_df["Patent_ID"].astype(str)
assert len(export_df) > 0, "export is empty — nothing survived the filters; check isApproved/status values"
assert not (_pid.isin(disapproved_ids) & (export_df["Section"] != "T1")).any(), \
    "disapproved patent leaked non-T1 rows into the export"
assert not (_pid.isin(exact_dup_ids)
            & export_df["Section"].isin(["T2", "G1", "M1", "M2", "M3"])).any(), \
    "exact duplicate (type 2) leaked its own label/T2 rows into the export"
_t2_blocks = set(zip(_pid[export_df["Section"] == "T2"],
                     export_df.loc[export_df["Section"] == "T2", "Sub_Dimension"].astype(str)))
assert _t2_blocks <= approved_blocks, "a non-approved T2 block leaked into the export"

format_review_workbook(export_df, OUTPUT_XLSX, truncate_long_text=False)

# ── Round-trip check: what 02b will read back must match what we exported ──
_rt = pd.read_excel(OUTPUT_XLSX, sheet_name="Review")
assert len(_rt) == len(export_df), \
    f"round-trip row-count mismatch: wrote {len(export_df)}, read back {len(_rt)}"
assert _rt["Patent_ID"].notna().all(), "round-trip lost Patent_ID values (merged cells?)"
assert list(_rt.columns) == list(export_df.columns), "round-trip changed the column set"

# ── Final labeller summary ──────────────────────────────────────────────────
print("\n" + "=" * 62)
print(f"  02a DONE — {sheet_name}")
print(f"  input : {REVIEWED_XLSX.name}  ({len(df_raw)} raw rows)")
print(f"  output: {OUTPUT_XLSX.name}")
print(f"  patents in export        : {export_df['Patent_ID'].nunique()}")
print(f"  approved image blocks    : {len(_t2_blocks)}")
print(f"  rows written             : {len(_rt)}  (round-trip verified)")
print(f"  disapproved patents (T1-only): {len(disapproved_ids)}")
print(f"  exact duplicates (label-less): {len(exact_dup_ids)}")
print("  → next: run 02b_postprocessing (it auto-picks this newest file).")
print("=" * 62)


scope enforcement: dropped 120 non-T1 row(s) for 60 disapproved patent(s) | dropped 0 T2/G1/M1-M3 row(s) for 167 exact-duplicate patent(s)
approved-only filter: kept 22684/25364 rows (301 approved image blocks, 1443 blocks dropped)
Successfully generated formatted workbook (Review + Compact): /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/reviewed xlsxs/Review_postprocess_Batch_01_20260706_141550.xlsx

  02a DONE — Batch_01
  input : reviewed_patents_Batch_01.xlsx  (25584 raw rows)
  output: Review_postprocess_Batch_01_20260706_141550.xlsx
  patents in export        : 393
  approved image blocks    : 301
  rows written             : 22684  (round-trip verified)
  disapproved patents (T1-only): 60
  exact duplicates (label-less): 167
  → next: run 02b_postprocessing (it auto-picks this newest file).
